In [ ]:
import os

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0,1")
os.environ.setdefault("OMP_NUM_THREADS", "2")

import sys
sys.path.append("/workspace/")

import re
import ast
import json
import time
import cv2
import multiprocessing as mp
from dotenv import load_dotenv
from huggingface_hub import login
from transformers import AutoProcessor
import resources.prompt as prompt_module

load_dotenv()
hf_token = os.getenv("HF_TOKEN")
if hf_token:
    login(token=hf_token)

TARGET_FPS = 5.0

In [ ]:
def build_prompt(fps, width, height):
    instruction = prompt_module.instruction
    return_format = prompt_module.return_format

    video_info = (
        f"The original video has a frame rate of {fps:.2f} frames per second. "
        f"Each original frame corresponds to {1000 / fps:.2f} milliseconds. "
        f"The video resolution is {width}x{height}. "
        f"Use this information to calculate the exact time and predict accurate bounding box coordinates."
    )

    return instruction + "\n" + return_format + "\n" + video_info

def default_result(center_time, center_x, center_y):
    # return {
    #     "time": center_time,
    #     "coordinate": [[center_x, center_y], [center_x, center_y]],
    #     "type": "rear-end",
    #     "reasoning": "invalid response format",
    # }
    return {
        "time": 0.0,
        "coordinate": [[0, 0], [0, 0]],
        "type": "rear-end",
        "reasoning": "invalid response format",
    }

def parse_in_json(llm_response, video_path):
    text = llm_response.strip()
    if not text.endswith("}"):
        if not text.endswith('"'):
            text += '"'
        text += "}"
    center_time = f"{(cv2.VideoCapture(video_path).get(cv2.CAP_PROP_FRAME_COUNT) / cv2.VideoCapture(video_path).get(cv2.CAP_PROP_FPS) / 2):.2f}"
    center_x = int(cv2.VideoCapture(video_path).get(cv2.CAP_PROP_FRAME_WIDTH) / 2)
    center_y = int(cv2.VideoCapture(video_path).get(cv2.CAP_PROP_FRAME_HEIGHT) / 2)
    try:
        temp = ast.literal_eval(text)
    except Exception:
        temp = default_result(center_time, center_x, center_y)
    if not isinstance(temp, dict):
        temp = default_result(center_time, center_x, center_y)
    # 만약 dicrionary에 collision_frame이 있다면 time을 계산해서 넣어주기
    if "collision_frame" in temp:
        try:
            collision_frame = temp["collision_frame"]
            fps = round(cv2.VideoCapture(video_path).get(cv2.CAP_PROP_FPS))
            center_time = collision_frame * 5
            temp["time"] = f"{center_time:.2f}"
        except Exception:
            # time을 전체 비디오의 중앙 값으로 넣어주기
            temp["time"] = f"{(cv2.VideoCapture(video_path).get(cv2.CAP_PROP_FRAME_COUNT) / fps / 2):.2f}"

    save_dir = os.path.join("result", "parsed_json")
    os.makedirs(save_dir, exist_ok=True)
    video_name = os.path.splitext(os.path.basename(video_path))[0]
    save_path = os.path.join(save_dir, f"{video_name}.json")

    try:
        with open(save_path, "w", encoding="utf-8") as f:
            json.dump(temp, f, ensure_ascii=False, indent=2)
    except Exception as e:
        print(f"failed to save parsed json for {video_path}: {e}", flush=True)

    return temp

In [ ]:
def load_video_frames(video_path, target_fps=TARGET_FPS):
    cap = cv2.VideoCapture(video_path)
    src_fps = cap.get(cv2.CAP_PROP_FPS)
    if src_fps <= 0:
        src_fps = target_fps

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_source_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    step = max(int(src_fps / target_fps), 1)

    frames = []
    sampled_indices = []
    idx = 0

    while True:
        if not cap.grab():
            break

        if idx % step != 0:
            idx += 1
            continue

        ret, frame = cap.retrieve()
        if not ret:
            break

        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = frame.astype("uint8", copy=False)
        frames.append(frame)
        sampled_indices.append(idx)
        idx += 1

    cap.release()

    metadata = {
        "src_fps": src_fps,
        "width": width,
        "height": height,
        "total_source_frames": total_source_frames,
        "sampled_indices": sampled_indices,
        "target_fps": target_fps
    }
    return frames, metadata

In [ ]:
class VideoInferenceVLM:
    def video_inference(self, video_path, prompt, max_new_tokens=128):
        raise NotImplementedError("Subclasses should implement this method.")


class Qwen3VLInference(VideoInferenceVLM):
    def __init__(self, model_id="Qwen/Qwen3-VL-8B-Instruct", max_encoder_cache_size=12288, tensor_parallel_size=2):
        from vllm import LLM
        from vllm import SamplingParams
        self.SamplingParams = SamplingParams
        self.processor = AutoProcessor.from_pretrained(model_id)
        self.max_encoder_cache_size = max_encoder_cache_size
        self.llm = LLM(
            model=model_id,
            tensor_parallel_size=tensor_parallel_size,
            dtype="float16",
            max_model_len=16384,
            max_num_seqs=1,
            gpu_memory_utilization=0.92,
            mm_encoder_tp_mode="data",
            limit_mm_per_prompt={"image": 0, "video": 1},
            mm_processor_cache_gb=1,
            enforce_eager=False,
        )

    def _build_llm_input(self, video_frames, metadata, prompt):
        total_num_sampled_frames = len(video_frames)

        if metadata["src_fps"] > 0 and metadata["total_source_frames"] > 0:
            duration = metadata["total_source_frames"] / metadata["src_fps"]
        else:
            duration = total_num_sampled_frames / metadata["target_fps"] if metadata["target_fps"] > 0 else 0.0

        messages = [
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": prompt,
                    },
                    {
                        "type": "video",
                    },
                ],
            }
        ]

        text_prompt = self.processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        return {
            "prompt": text_prompt,
            "multi_modal_data": {
                "video": (
                    video_frames,
                    {
                        "fps": metadata["target_fps"],
                        "total_num_frames": metadata["total_source_frames"] if metadata["total_source_frames"] > 0 else total_num_sampled_frames,
                        "duration": duration,
                        "video_backend": "opencv",
                        "frames_indices": metadata["sampled_indices"] if metadata["sampled_indices"] else list(range(total_num_sampled_frames)),
                        "do_sample_frames": False,
                    },
                ),
            },
        }

    def _trim_video_to_frame_count(self, video_frames, metadata, keep_count):
        keep_count = max(1, min(keep_count, len(video_frames)))
        trimmed_frames = video_frames[:keep_count]
        trimmed_metadata = dict(metadata)
        if metadata.get("sampled_indices"):
            trimmed_metadata["sampled_indices"] = metadata["sampled_indices"][:keep_count]
        return trimmed_frames, trimmed_metadata

    def _handling_cache_overflow(self, error, video_path, video_frames, metadata):
        error_text = str(error)

        if "exceeds the pre-allocated encoder cache size" not in error_text:
            raise error

        match = re.search(
            r"video item with length (\d+), which exceeds .* size (\d+)",
            error_text,
        )
        if not match:
            raise error

        current_length = int(match.group(1))
        cache_limit = int(match.group(2))
        current_frame_count = len(video_frames)

        if current_frame_count <= 1:
            raise error

        keep_count = int(current_frame_count * cache_limit / current_length) - 1
        keep_count = max(1, min(keep_count, current_frame_count - 1))

        print(
            f"[trim] {os.path.basename(video_path)}: cache overflow "
            f"({current_length} > {cache_limit}), trimming frames "
            f"{current_frame_count} -> {keep_count}",
            flush=True,
        )

        video_frames, metadata = self._trim_video_to_frame_count(
            video_frames,
            metadata,
            keep_count,
        )

        return video_frames, metadata

    def video_inference(
        self,
        video_path,
        prompt,
        max_new_tokens=128,
        video_frames=None,
        metadata=None,
    ):
        if video_frames is None or metadata is None:
            video_frames, metadata = load_video_frames(video_path, target_fps=TARGET_FPS)

        sampling_params = self.SamplingParams(
            temperature=0.0,
            max_tokens=max_new_tokens,
        )

        while True:
            try:
                llm_input = self._build_llm_input(video_frames, metadata, prompt)
                outputs = self.llm.generate(
                    llm_input,
                    sampling_params=sampling_params,
                    use_tqdm=False,
                )
                return [outputs[0].outputs[0].text]

            except ValueError as e:
                video_frames, metadata = self._handling_cache_overflow(
                    e,
                    video_path,
                    video_frames,
                    metadata,
                )

inference = Qwen3VLInference()

In [ ]:
import cv2

instruction = prompt_module.instruction
return_format = prompt_module.return_format

video_path = "/workspace/dataset/videos/ZsPhpxb8OTM_00.mp4"
# video_path = "/workspace/dataset/videos/-2UPLUV7JLg_00.mp4"
# video_path = "/workspace/dataset/videos/-9oifpjUxxM_00.mp4"
# video_path = "/workspace/dataset/videos/zSujHC7MOcU_00.mp4"
cap = cv2.VideoCapture(video_path)
fps = round(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()
print(f"Video FPS: {fps}")
print(f"Video resolution: {width}x{height}")
video_info = f"The video has a frame rate of {fps} frames per second. It means that each indexed frame corresponds to {1000/fps:.2f} milliseconds. " \
    f"Use this information to calculate the exact time. " \
    f"The video resolution is {width}x{height}. Use this information to predict accurate bounding box coordinates. " \
    
prompt = instruction + "\n" + video_info + "\n" + return_format

start = time.time()
output = inference.video_inference(video_path, prompt, max_new_tokens=512)
print(output[0])
end = time.time()
print(f"Inference time: {end - start:.2f} seconds")

In [ ]:
from pprint import pprint
extracted_info = parse_in_json(output[0], video_path)
pprint(extracted_info)
# 비디오에서 모델이 예측한 프레임 불러오기
video = cv2.VideoCapture(video_path)
frame_index = extracted_info.get("collision_frame", 0) * 5

ret = False
frame = None
for _ in range(frame_index + 1):
    ret, frame = video.read()
    if not ret:
        break

# 프레임에 모델이 예측한 바운딩 박스 그리기
if ret:
    x1, y1 = extracted_info["coordinate"][0]
    x2, y2 = extracted_info["coordinate"][1]
    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
    cv2.putText(frame, f"Frame: {frame_index}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    # 결과 프레임을 이미지로 저장
    result_image_path = os.path.join("result", "frames", f"{os.path.splitext(os.path.basename(video_path))[0]}_frame_{frame_index}.jpg")
    os.makedirs(os.path.dirname(result_image_path), exist_ok=True)

    # 결과 프레임에 type과 time 정보 추가
    time_info = extracted_info.get("time", "N/A")
    type_info = extracted_info.get("type", "N/A")
    cv2.putText(frame, f"Time: {time_info}s", (10, 110), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.putText(frame, f"Type: {type_info}", (10, 150), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.imwrite(result_image_path, frame)
    print(f"Result frame with time and type info saved at: {result_image_path}")

    # 결과 프레임을 화면에 표시 (주피터 노트북 셀)
    from PIL import Image
    display(Image.open(result_image_path))